In [1]:
import pandas as pd

order_fact = pd.read_parquet("../data_processed/order_fact.parquet")

# 只保留已完成订单
order_fact = order_fact[order_fact["order_status"] == "delivered"].copy()

order_fact.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,gmv,item_cnt,payment_total,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,38.71,1.0,38.71,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,141.46,1.0,141.46,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,179.12,1.0,179.12,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,72.20,1.0,72.20,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,28.62,1.0,28.62,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,2018-02


定义分析时间

In [2]:
snapshot_date = order_fact["order_purchase_timestamp"].max()
snapshot_date

Timestamp('2018-08-29 15:00:37')

构建rfm指标

In [3]:
rfm = (
    order_fact
    .groupby("customer_unique_id")
    .agg(
        Recency=("order_purchase_timestamp",
                 lambda x: (snapshot_date - x.max()).days),
        Frequency=("order_id", "nunique"),
        Monetary=("gmv", "sum")
    )
    .reset_index()
)

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19
2,0000f46a3911fa3c0805444483337064,536,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89


分位数打分

In [4]:
rfm["R_score"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1])
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5])
rfm["M_score"] = pd.qcut(rfm["Monetary"], 5, labels=[1,2,3,4,5])

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary,R_score,F_score,M_score
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,4,1,4
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,4,1,1
2,0000f46a3911fa3c0805444483337064,536,1,86.22,1,1,2
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,2,1,1
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,2,1,4


组合rfm分数

In [5]:
rfm["RFM_score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

rfm.head()

,customer_unique_id,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,4,1,4,414
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,4,1,1,411
2,0000f46a3911fa3c0805444483337064,536,1,86.22,1,1,2,112
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,2,1,4,214


看高价值用户比例

In [6]:
high_value = rfm[
    (rfm["R_score"].astype(int) >= 4) &
    (rfm["F_score"].astype(int) >= 4) &
    (rfm["M_score"].astype(int) >= 4)
]

len(high_value) / len(rfm)

0.069592322029178

In [7]:
rfm.head()

,customer_unique_id,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,4,1,4,414
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,4,1,1,411
2,0000f46a3911fa3c0805444483337064,536,1,86.22,1,1,2,112
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,2,1,4,214


用户价值结构

分布

In [8]:
rfm.describe()

,Recency,Frequency,Monetary
count,93358.000000,93358.000000,93358.000000
mean,236.941773,1.033420,165.168210
std,152.591453,0.209097,226.292101
min,0.000000,1.000000,9.590000
25%,113.000000,1.000000,63.010000
50%,218.000000,1.000000,107.780000
75%,345.000000,1.000000,182.510000
max,713.000000,15.000000,13664.080000


消费集中度

In [9]:
total_revenue = rfm["Monetary"].sum()

rfm_sorted = rfm.sort_values("Monetary", ascending=False)

rfm_sorted["cum_revenue_pct"] = rfm_sorted["Monetary"].cumsum() / total_revenue

rfm_sorted.head()

,customer_unique_id,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_score,cum_revenue_pct
3724,0a0a92112bd4c708ca5fde585afaa872,333,1,13664.08,2,1,5,215,0.000886
79636,da122df9eeddfedc1dc1f5349a1a690c,514,2,7571.63,1,5,5,155,0.001377
43168,763c8b1c9c68a0229c42c9fc6f662b93,45,1,7274.88,5,3,5,535,0.001849
80463,dc4802a71eae9be1dd28f5d788ceb526,562,1,6929.31,1,5,5,155,0.002298
25436,459bef486812aa25204be022145caa62,34,1,6922.21,5,2,5,525,0.002747


前20%用户贡献

In [10]:
top_20_pct_index = int(len(rfm_sorted) * 0.2)

top_20_revenue_share = (
    rfm_sorted.iloc[:top_20_pct_index]["Monetary"].sum()
    / total_revenue
)

top_20_revenue_share

0.5352439798281736

划分用户类型

In [11]:
rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)

rfm["segment"] = "Other"

rfm.loc[
    (rfm["R_score"] >= 4) &
    (rfm["F_score"] >= 4) &
    (rfm["M_score"] >= 4),
    "segment"
] = "High Value"

rfm.loc[
    (rfm["R_score"] <= 2) &
    (rfm["F_score"] <= 2),
    "segment"
] = "Low Value"

rfm["segment"].value_counts(normalize=True)

segment
Other         0.769886
Low Value     0.160522
High Value    0.069592
Name: proportion, dtype: float64